# True-feature OLS: saved compact report

Reader only: **Run All never generates, recovers, calibrates, or fits worlds**. The accepted
compact report is included; an absent file means **NO RESULT**, not zero recovery. Science is
not attempted by this notebook. Whole-run scientific acceptance and coordinator release
remain separate gates.

Use the existing project environment (no installation). From the repository or notebook
directory, the default is notebook-relative `data/linear-recovery-true-feature-sweep-summary.json`.
For a copied notebook started elsewhere, set `NOTEBOOK_DIRECTORY` to its directory, or supply
`SUMMARY_PATH` explicitly. Relative explicit paths are relative to that notebook directory.
The accepted reader module must already be importable (for example via the checkout's
`PYTHONPATH`). This is location portability on the accepted POSIX reader, not Windows support.

The default trusted digest is the independently accepted report SHA-256 below. Change it only
when independently accepted bytes are intentionally supplied; setting it to `None` refuses
a present report. It is never computed from the file being checked. The tiny temporary filename
adapter preserves exact bytes and allows the frozen reader to run without writing any result into the checkout; it never opens a raw archive.

In [ ]:
from pathlib import Path

# Optional reader inputs; no scientific execution settings.
NOTEBOOK_DIRECTORY = None
SUMMARY_PATH = None
EXPECTED_SUMMARY_SHA256 = "d8b6ce0d4c91e6f3da864103e634a7c33ecfba89b9837f83707e9ab49da613c8"

In [ ]:
import hashlib
import json
import os
import re
import stat
from html import escape
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import HTML, display

from scripts.linear_recovery_true_feature_sweep import (
    _portable_report_parent,
    read_compact_summary,
)

if NOTEBOOK_DIRECTORY is not None:
    notebook_directory = Path(NOTEBOOK_DIRECTORY).absolute()
elif (Path.cwd() / "docs/examples/linear-recovery-true-feature-sweep.ipynb").is_file():
    notebook_directory = Path.cwd() / "docs/examples"
else:
    notebook_directory = Path.cwd()
summary_path = (
    Path(SUMMARY_PATH)
    if SUMMARY_PATH is not None
    else Path("data/linear-recovery-true-feature-sweep-summary.json")
)
if not summary_path.is_absolute():
    summary_path = notebook_directory / summary_path
summary = None
validation = "not_attempted"
try:
    summary_path.lstat()
except FileNotFoundError:
    if SUMMARY_PATH is not None:
        raise FileNotFoundError(f"Explicit summary path is missing: {summary_path}")
    print(
        "NO RESULT: no accepted compact summary is available. Science: not attempted by this reader."
    )
else:
    if (
        not isinstance(EXPECTED_SUMMARY_SHA256, str)
        or re.fullmatch(r"[0-9a-f]{64}", EXPECTED_SUMMARY_SHA256) is None
    ):
        raise ValueError(
            "Present summary requires EXPECTED_SUMMARY_SHA256 from independent acceptance."
        )
    with _portable_report_parent(summary_path.parent) as (parent, _):
        fd = os.open(summary_path.name, os.O_RDONLY | os.O_NOFOLLOW | os.O_NONBLOCK, dir_fd=parent)
        with os.fdopen(fd, "rb") as stream:
            if not stat.S_ISREG(os.fstat(stream.fileno()).st_mode):
                raise ValueError("Summary must be a regular file.")
            compact_bytes = stream.read()
    if hashlib.sha256(compact_bytes).hexdigest() != EXPECTED_SUMMARY_SHA256:
        raise ValueError("Summary SHA256 mismatch: refusing untrusted bytes.")
    reference = {
        "version": "ols-report-evidence/v1",
        "kind": "summary",
        "locator": f"summary-{EXPECTED_SUMMARY_SHA256}.json",
        "sha256": EXPECTED_SUMMARY_SHA256,
    }
    with TemporaryDirectory(prefix="ols-compact-reader-") as temporary:
        (Path(temporary) / reference["locator"]).write_bytes(compact_bytes)
        checked = read_compact_summary(temporary, reference)
    summary = checked["summary"]
    validation = checked["validation"]
    print(
        f"Validation: {validation}; trusted SHA256: {EXPECTED_SUMMARY_SHA256}; {len(compact_bytes)} bytes."
    )
    print("Raw-science verification: not attempted. Compact integrity is not execution proof.")

## Scope and methods — current all-104 policy

Scalar-baseline restricted population: relative outcome noise, intrinsic baseline constant
`b`, random-walk baseline standard deviation zero, baseline floor null. Not prevalence for
unrestricted C2 random-walk baselines. Retained generator axes and numerical thresholds are
in the actual saved configuration below; no thresholds are selected from these results.

`D_C = {k: g_cy[k]=1}`, `D_Z = {m: g_zy[m]=1}`. In that order,
`X = [observed-path φ[D_C], Z[D_Z], 1]`, `θ = [beta[D_C], rho_zy[D_Z], b]`.
Each feature uses **actual adstock then actual saturation**, with saved saturation scale and
full supplied audit history before row selection; never divide contributions by beta or
recompute saturation scale from the reported window. Upstream/indirect effects remain inside
observed-path features; feeder-only treatments have no direct beta estimand.

With `u = sum(direct latent contributions)`, the paired targets are exactly
**`y0 = outcome - u - outcome_noise`** and **`y1 = outcome - u`**.
There is no indirect-effect subtraction and no synthetic noise. Both fits share X, coefficient
order and rows. Raw observed outcome, when retained, is separately labelled unadjusted and
may be misspecified; it is not a paired recovery target.

**Current owner policy: all 104 rows 0–103 in every fit/recovery; generator B=0 and
transformation Wdiscard=0. Startup transients are INCLUDED, not excluded.** The future
transformation-warmup correction is Issue 42, not applied here. The saved `warmup` field denotes
generator prehistory B, not a separate exclusion. No 97-row claim is made.

The accepted compact result covers 139 conforming worlds; 861 allocation-excluded worlds
were not run. The separate earlier 145-world preliminary archive is nonconforming and is not
aggregated. Compact construction took 1815.960578 seconds; the final receipt took
1831.415623 seconds including persistence/read verification, with 52.940351 seconds
finalization. The observed noisy outcomes are available for 139 worlds, not exact recovery.
Startup transients remain included; the Issue 42 correction is deferred.

These are structural recovery diagnostics, not causal identification.

In [ ]:
if summary is not None:
    contract = summary["evidence"]["input"]["contract"]
    provenance = {
        "summary_version": summary["version"],
        "input_version": contract["version"],
        "bindings": summary["bindings"],
        "source_commit": contract["source_commit"],
        "source_hashes": contract["source_hashes"],
        "catalog_schema_path": contract["catalog_schema_path"],
        "catalog_schema_sha256": contract["catalog_schema_sha256"],
        "saved_runtime_environment": contract["environment"],
        "seed_manifest_digest": contract["seed_manifest_digest"],
        "required_order": contract["required_order"],
        "config": contract["config"],
        "allocation_inputs": summary["evidence"]["allocation"]["inputs"],
        "allocation_formula": summary["evidence"]["allocation"]["formula"],
        "runtime": summary["runtime"],
    }
    display(
        HTML(
            "<h3>Actual saved configuration and provenance</h3><pre>"
            + escape(json.dumps(provenance, indent=2))
            + "</pre>"
        )
    )
    print("Saved runtime paths are provenance strings only; this reader does not use them.")
    for phase, records in contract["seed_manifest"].items():
        display(
            HTML(
                f"<details><summary>Frozen {escape(phase)} seed manifest</summary>"
                + pd.DataFrame(records).to_html(index=False)
                + "</details>"
            )
        )

## World-unit accounting and uncertainty

`Ncap` is the frozen allocated prefix of maximum 1000; `A` actually started; `G` has retained
verified generation/graph evidence; `E` is truth/identity eligible; `F` is full-rank eligible;
`R` is noisefree recovered within F. Conservation: `1000 = allocation_excluded + Ncap`,
`Ncap = A + runtime_unattempted`, `A = G + confirmed_generation_failed + evidence_unavailable`,
`G = E + generated_ineligible`, `E = F + rank_deficient`. Missing-evidence substatuses distinguish
known completed generation from unknown generation outcome. Analysis/persistence/timeout
stages overlap and must not be added to that disjoint partition.

Every rate names numerator, denominator and population: full rank/E, rank deficiency/E,
noisefree recovery/F, paired-noisy availability/E, failures/A, graph patterns/G. Wilson 95%
intervals are descriptive, non-simultaneous, one binary indicator per independently seeded
**world** — never rows, coefficients, or the two paired targets as replicates. Runtime dropout
limits representativeness. A null rate/interval means unavailable, not zero. The pooled
400 flag is allocation adequacy only, **not an execution stop** or subgroup precision claim.
The minimum 10 applies **only to conditional outcome rates**; zero/sparse frequency cells
and their world-unit Wilson intervals are retained.

In [ ]:
if summary is not None:
    display(
        HTML(
            "<h3>Conserved counts</h3>"
            + pd.DataFrame(
                list(summary["counts"].items()), columns=["population", "worlds"]
            ).to_html(index=False)
        )
    )
    display(
        HTML(
            "<h3>Precision flags</h3><pre>"
            + escape(json.dumps(summary["precision"], indent=2))
            + "</pre>"
        )
    )
    for section in (
        "rates",
        "failure_frequencies",
        "stage_failure_frequencies",
        "truth_failure_frequencies",
        "conditioning_frequencies",
    ):
        rows = [{"measure": key, **value} for key, value in summary[section].items()]
        display(HTML(f"<h3>{escape(section)}</h3>" + pd.DataFrame(rows).to_html(index=False)))
    display(
        HTML(
            "<details><summary>Every started attempt: states, missing evidence and reasons</summary>"
            + pd.json_normalize(summary["attempts"]).to_html(index=False)
            + "</details>"
        )
    )

## Natural frequencies and separate conditioning

T and M were drawn independently uniformly on 1–10, not balanced/forced into strata.
Planned configuration frequencies use Ncap; attempted frequencies use A. The exact 100 cells
and nine coarse 1–3 / 4–7 / 8–10 band pairs are retained, including zeros. Direct dimensions
and all 256 edge-family presence patterns use G, **conditional on retained verified generation**;
missing evidence is not an inferred graph pattern. Conditioning (easy/moderate/hard) is separate
from full rank versus rank deficiency. Raw and standardized condition diagnostics remain
separate; standardization excludes the intercept until explicit ones are appended.

In [ ]:
if summary is not None:
    for population in ("planned_configured", "attempted_configured"):
        for group, records in summary[population].items():
            display(
                HTML(
                    f"<details><summary>{escape(population)} / {escape(group)}</summary>"
                    + pd.json_normalize(records).to_html(index=False)
                    + "</details>"
                )
            )
    for name, dimension in summary["direct_dimensions"].items():
        exact = [{"dimension": i, **rate} for i, rate in enumerate(dimension["exact"])]
        display(
            HTML(
                f"<h3>Direct dimension {escape(name)} / G</h3>"
                + pd.DataFrame(exact).to_html(index=False)
                + pd.json_normalize(dimension["bins"]).to_html(index=False)
            )
        )
    print("Pattern bit order:", summary["graph_patterns"]["bit_order"])
    patterns = [
        {"pattern": key, **rate} for key, rate in summary["graph_patterns"]["frequencies"].items()
    ]
    display(
        HTML(
            "<details><summary>All 256 patterns / G (including zero counts)</summary>"
            + pd.DataFrame(patterns).to_html(index=False)
            + "</details>"
        )
    )

## Paired diagnostics and nine pre-fit examples

Noisy availability is not promised coefficient recovery. Coefficient errors are descriptive
world-level summaries over F (direct beta/rho families only where present), without binomial
intervals or coefficient/row pseudoreplication. Rank-deficient prediction residuals remain
visible over E, but do not identify coefficients. Nonfinite/unavailable condition values are
not replaced. The first scheduled example in each band was selected **before fits**; absent,
unattempted, failed or ineligible selections are never replaced by successful worlds.

The compact examples show saved truth, parameters, dimensions, identities, coefficients and
paired-fit diagnostics only. **Full trajectories, designs and target arrays are not present in
the compact report**: no fabricated trajectory plots. Raw science verification requires a
separate authorized check of the original archive; this notebook does not perform it.

In [ ]:
if summary is not None:
    display(
        HTML(
            "<h3>Descriptive world-level diagnostics (not binomial rates)</h3><pre>"
            + escape(json.dumps(summary["descriptive"], indent=2))
            + "</pre>"
        )
    )
    overview = [
        {
            key: value
            for key, value in example.items()
            if key not in ("details", "graph", "references", "catalog")
        }
        for example in summary["examples"]
    ]
    display(
        HTML(
            "<h3>All nine frozen selections</h3>" + pd.json_normalize(overview).to_html(index=False)
        )
    )
    for example in summary["examples"]:
        label = f"Attempt {example['attempt']}: {example['availability_reason']}"
        display(
            HTML(
                f"<details><summary>{escape(label)}</summary><pre>"
                + escape(json.dumps(example, indent=2))
                + "</pre></details>"
            )
        )